In [0]:
import os, json, sys, time, mlflow, traceback
from pathlib import Path
from mlflow.tracking import MlflowClient
from pyspark.sql import functions as F

sys.path.append("/Workspace/9900-f18a-cake") 
sys.path.append("/Workspace/9900-f18a-cake/mt-method2/src")
sys.path.append("/Workspace/9900-f18a-cake/mt-method2")
from mch.models.training import BatchModelTrainer

In [0]:
def _get_param(name: str, default: str = "") -> str:
    try:
        return dbutils.widgets.get(name) or default
    except Exception:
        import os
        return os.getenv(name, default)


In [0]:
def _safe_name(s: str) -> str:
    """Make a filesystem/MLflow artifact-safe subdir name."""
    return "".join(ch if ch.isalnum() or ch in "-_" else "_" for ch in str(s))

In [0]:
NODE_ID   = _get_param("NODE_ID",   "ZERO2")
ONLY_NODE = _get_param("ONLY_NODE", "")
SAVE_MODE = _get_param("SAVE_MODE", "nosave")  # allow "save"/"nosave" or true/false
OUT_DIR   = _get_param("OUT_DIR", "/Volumes/cb_prod/comp9300-9900-f18a-cake/mt_method2_models")

In [0]:
def _set_env_from_plan(plan: dict) -> None:
    import os

    # Optional knobs – keep your existing ones too
    if "prefilter_chunk_size" in plan:
        os.environ["MCH_PREFILTER_CHUNK_SIZE"] = str(plan["prefilter_chunk_size"])
    os.environ["MCH_DISABLE_DM"] = "1" if plan.get("disable_dm", True) else "0"

    only_node = (plan.get("only_node") or "").strip()
    # Set BOTH names so whichever your Training code reads will work
    if only_node:
        os.environ["MCH_ONLY_NODE"] = only_node
        os.environ["ONLY_NODE"]     = only_node
    else:
        os.environ.pop("MCH_ONLY_NODE", None)
        os.environ.pop("ONLY_NODE", None)


In [0]:
def _print_result_summary(result: dict, default_node_id: str = "") -> None:
    node = result.get("node_id") or default_node_id or "UNKNOWN"
    print(f"\n=== RESULTS for node: {node} ===")
    if result.get("error"):
        print("Status : FAILED")
        print("Error  :", result["error"])
        if result.get("trace"):
            print("\nTraceback:")
            print(result["trace"])
        return

    print("Status : OK")
    metrics = result.get("metrics") or {}
    if metrics:
        w = max(len(k) for k in metrics.keys())
        for k in sorted(metrics.keys()):
            v = metrics[k]
            try:
                v = float(v)
                print(f"{k.rjust(w)} : {v:.4f}")
            except Exception:
                print(f"{k.rjust(w)} : {v}")
    else:
        print("No metrics were logged (possibly skipped due to sample count or single subgroup).")

    if result.get("t_sec") is not None:
        print(f"Elapsed (sec) : {float(result['t_sec']):.2f}")

    try:
        run = mlflow.active_run()
        if run:
            print("MLflow run id :", run.info.run_id)
    except Exception:
        pass


In [0]:
def run_child(plan: dict) -> dict:
    """
    Train for a single node, optionally saving the model(s) and logging them to MLflow.

    plan keys used:
      - node_id: str
      - only_node: str|None  (to force training only that node inside your trainer)
      - save: "save" | "nosave"
      - out_dir: str (dbfs:/... or /dbfs/...)
      - mlflow_experiment_path: optional str (will set the experiment if provided)
      - prefilter_* / rf_n_jobs / cv_n_jobs / disable_dm ... (forwarded via env by your helper)
    """
    start_ts = time.time()
    node_id   = plan.get("node_id", "UNKNOWN")
    only_node = plan.get("only_node")
    raw_save = plan.get("save", "nosave")
    save_flag = (raw_save is True) or (str(raw_save).lower() in {"save", "true", "1", "yes"})
    out_dir   = plan.get("out_dir", OUT_DIR)
    exp_path  = plan.get("mlflow_experiment_path")  # optional

    # Set/ensure MLflow experiment if provided
    if exp_path:
        mlflow.set_experiment(exp_path)

    # Configure trainer via env vars (your existing helper)
    _set_env_from_plan(plan)

    tags = {
        "orchestrator": "child_notebook",
        "node_id": node_id,
        "only_node": str(only_node),
        "disable_dm": str(plan.get("disable_dm", True)),
        "save_flag": str(save_flag),
    }
    params = {
        "prefilter_topk":       plan.get("prefilter_topk", 200),
        "prefilter_scan_max":   plan.get("prefilter_scan_max", 20000),
        "prefilter_chunk_size": plan.get("prefilter_chunk_size", 5000),
        "rf_n_jobs":            plan.get("rf_n_jobs", 1),
        "cv_n_jobs":            plan.get("cv_n_jobs", 1),
    }

    result = {
        "ok": False,
        "node_id": node_id,
        "metrics": {},
        "error": None,
        "trace": None,
        "t_sec": None,
    }

    # Prepare save directory (per-node) if saving
    node_subdir = None
    if save_flag:
        safe_node = _safe_name(only_node or node_id)
        # Allow both dbfs:/ and /dbfs/ inputs; normalize to local /dbfs path for Python IO
        if out_dir.startswith("dbfs:/"):
            local_root = "/dbfs" + out_dir[len("dbfs:"):]   # e.g. dbfs:/tmp/... -> /dbfs/tmp/...
        else:
            local_root = out_dir
        node_subdir = os.path.join(local_root, safe_node)
        os.makedirs(node_subdir, exist_ok=True)

    with mlflow.start_run(run_name=f"node={node_id}"):
        mlflow.set_tags(tags)
        mlflow.log_params(params)

        try:
            trainer = BatchModelTrainer()
            # Your trainer should write artifacts when save_dir is not None.
            stats = trainer.train_all_models(
                save_dir=node_subdir if save_flag else None,
                raise_on_error=False
            )

            node_key = only_node if only_node else node_id
            node_stats = stats.get(node_key, {}) if isinstance(stats, dict) else {}
            metrics = node_stats.get("metrics", {}) if isinstance(node_stats, dict) else {}

            # Log numeric metrics
            for k, v in metrics.items():
                try:
                    mlflow.log_metric(k, float(v))
                except Exception:
                    pass  # ignore non-numeric

            # If we saved, log the saved files to MLflow under models/<node_id>/
            if save_flag and node_subdir and os.path.exists(node_subdir):
                # Try common filenames first; if none found, log the whole folder.
                candidates = [
                    "model.joblib", "model.pkl", "pipeline.pkl",
                    "model.bin", "model.pt"
                ]
                logged_any = False
                for c in candidates:
                    p = os.path.join(node_subdir, c)
                    if os.path.exists(p):
                        mlflow.log_artifact(p, artifact_path=f"models/{_safe_name(node_id)}")
                        logged_any = True
                if not logged_any:
                    # Fall back: log everything in the node folder
                    for root, _, files in os.walk(node_subdir):
                        for f in files:
                            mlflow.log_artifact(
                                os.path.join(root, f),
                                artifact_path=f"models/{_safe_name(node_id)}"
                            )

            result.update(ok=True, metrics=metrics)

        except Exception as e:
            result["error"] = f"{type(e).__name__}: {e}"
            result["trace"] = traceback.format_exc()

        finally:
            result["t_sec"] = round(time.time() - start_ts, 2)

    return result

In [0]:
plan = {
    "node_id":   NODE_ID,
    "only_node": ONLY_NODE,
    "save":      SAVE_MODE,   # "save" or "nosave" (also works with true/false now)
    "out_dir":   OUT_DIR,
}
res = run_child(plan)
_print_result_summary(res, default_node_id=NODE_ID)
print("Child done:", NODE_ID)